# Insulin response predictor — guided run

This notebook runs the same tested pipeline as the CLI and displays its results inline. It starts with fictional data. **Research only: no output is a treatment instruction.**

In [ ]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if (Path.cwd().parent / 'pyproject.toml').exists() else Path.cwd()
%pip install -q -e "$PROJECT_ROOT"

## 1. Settings
Leave MODE as fictional for the complete safe demo. Change it to csv only after placing the four sheet exports in an ignored local directory and creating config/subject.yaml.

In [ ]:
MODE = 'fictional'  # 'fictional' or 'csv'
INPUT_DIRECTORY = PROJECT_ROOT / 'data' / 'raw' / 'latest'
OUTPUT_DIRECTORY = PROJECT_ROOT / 'notebook-output'
SUBJECT_CONFIG = PROJECT_ROOT / 'config' / 'subject.yaml'
FICTIONAL_DAYS = 90
FICTIONAL_SEED = 42

## 2. Load data and validate

In [ ]:
import pandas as pd
from IPython.display import Markdown, display
from insulin_response_predictor.configuration import load_policy_config
from insulin_response_predictor.io import load_csv_exports
from insulin_response_predictor.pipeline import run_pipeline
from insulin_response_predictor.policy import PolicyConfig
from insulin_response_predictor.synthetic import generate_synthetic_dataset
from insulin_response_predictor.validation import validate_dataset

if MODE == 'fictional':
    tables = generate_synthetic_dataset(days=FICTIONAL_DAYS, seed=FICTIONAL_SEED, scenario='identifiable')
    policy_config = PolicyConfig()
elif MODE == 'csv':
    tables = load_csv_exports(INPUT_DIRECTORY)
    policy_config = load_policy_config(SUBJECT_CONFIG)
else:
    raise ValueError("MODE must be 'fictional' or 'csv'")

issues = validate_dataset(tables)
issue_table = pd.DataFrame([issue.__dict__ for issue in issues])
display(issue_table if not issue_table.empty else Markdown('**Validation passed with no issues.**'))
if any(issue.severity == 'error' for issue in issues):
    raise ValueError('Fix validation errors before continuing.')

## 3. Run the complete pipeline
The gate is automatic. A weak forward model stops the policy experiment.

In [ ]:
manifest = run_pipeline(tables, OUTPUT_DIRECTORY, policy_config=policy_config)
display(pd.DataFrame(manifest['stages']).T)
print('Final status:', manifest['status'])

## 4. Review reports and charts

In [ ]:
from IPython.display import Image

for report in [
    OUTPUT_DIRECTORY / '01_data_quality' / 'data_quality.md',
    OUTPUT_DIRECTORY / '02_exploratory_analysis' / 'eda_report.md',
    OUTPUT_DIRECTORY / '03_forward_model' / 'forward_report.md',
    OUTPUT_DIRECTORY / '04_policy_experiment' / 'policy_report.md',
]:
    if report.exists():
        display(Markdown(report.read_text(encoding='utf-8')))

for chart in [
    OUTPUT_DIRECTORY / '02_exploratory_analysis' / 'meal_response.png',
    OUTPUT_DIRECTORY / '03_forward_model' / 'predicted_vs_actual.png',
    OUTPUT_DIRECTORY / '03_forward_model' / 'residual_diagnostics.png',
]:
    if chart.exists():
        display(Markdown(f'### {chart.stem.replace("_", " ").title()}'))
        display(Image(filename=str(chart)))

## Interpretation boundary
A PASS means only that retrospective experimentation met the configured statistical gate. It does not validate a dose, make a prescription, or establish safety for real-world use.